# CarbonX ML Service - Example Usage

This notebook demonstrates how to use the ML forecasting models for carbon emissions prediction.

**Setup:** Install dependencies first:
```bash
pip install -r requirements.txt jupyter
```

## 1. Generate Synthetic Test Data

Create realistic 36-month historical data with trend and seasonality

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# Generate 36 months (3 years) of synthetic data
np.random.seed(42)
months = 36
historical_data = []

for month in range(months):
    # Base emissions: 200 kg/month
    base = 200
    
    # Upward trend: +2 kg/month
    trend = month * 2
    
    # Seasonality: Peak in summer (June-August) due to AC usage
    # Monthly pattern over 12 months
    seasonal_amplitude = 50
    seasonal = seasonal_amplitude * np.sin(month * 2 * np.pi / 12 - np.pi/2)
    
    # Random noise: ±5% variation
    noise = np.random.normal(0, base * 0.05)
    
    total = base + trend + seasonal + noise
    total = max(100, total)  # Ensure positive
    
    historical_data.append(round(total, 2))

print(f"Generated {len(historical_data)} months of data")
print(f"Range: {min(historical_data):.1f} - {max(historical_data):.1f} kg CO2e/month")
print(f"Mean: {np.mean(historical_data):.1f} kg CO2e/month")

# Visualize
plt.figure(figsize=(12, 4))
plt.plot(historical_data, 'b-', linewidth=2)
plt.title('Synthetic 36-Month Historical Emissions Data')
plt.xlabel('Month')
plt.ylabel('Emissions (kg CO2e)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nFirst 12 months: {historical_data[:12]}")
print(f"Last 12 months: {historical_data[-12:]}")

## 2. Test Individual Models

### Prophet Model

In [ ]:
from services.prophet_model import ProphetForecaster

# Initialize and train
prophet = ProphetForecaster(interval_width=0.9)
print("Training Prophet model...")
result = prophet.train(historical_data, start_date='2023-01-01')

print(f"Status: {result['status']}")
print(f"Training time: {result['training_time']:.2f}s")
print(f"\nMetrics:")
for metric, value in result['metrics'].items():
    print(f"  {metric.upper()}: {value}")

# Generate forecast
print(f"\nGenerating 12-month forecast...")
forecast = prophet.forecast_future(periods=12)

# Display forecast
forecast_df = pd.DataFrame(forecast)
print(forecast_df.to_string())

### XGBoost Model

In [ ]:
from services.xgboost_model import XGBoostForecaster

# Generate employee counts (correlate with emissions)
employee_counts = [100 + (i * 2) for i in range(36)]

# Initialize and train
xgb = XGBoostForecaster(n_estimators=100, max_depth=5, learning_rate=0.1)
print("Training XGBoost model...")
result = xgb.train(historical_data, employee_counts)

print(f"Status: {result['status']}")
print(f"Training time: {result['training_time']:.2f}s")
print(f"\nMetrics:")
for metric, value in result['metrics'].items():
    print(f"  {metric.upper()}: {value}")

# Feature importance
print(f"\nTop 10 Most Important Features:")
feature_importance = xgb.get_feature_importance()
for i, (feature, importance) in enumerate(feature_importance[:10], 1):
    print(f"  {i}. {feature}: {importance:.4f}")

# Generate forecast
print(f"\nGenerating 12-month forecast...")
forecast = xgb.forecast_future(historical_data, periods=12, employee_counts=employee_counts)

forecast_df = pd.DataFrame(forecast)
print(forecast_df.to_string())

### Holt-Winters Model

In [ ]:
from services.holt_winters_model import HoltWintersForecaster

# Initialize and train
hw = HoltWintersForecaster(alpha=0.3, beta=0.1, gamma=0.1)
print("Training Holt-Winters model...")
result = hw.train(historical_data, start_date='2023-01-01')

print(f"Status: {result['status']}")
print(f"Training time: {result['training_time']:.4f}s")
print(f"\nMetrics:")
for metric, value in result['metrics'].items():
    print(f"  {metric.upper()}: {value}")

# Generate forecast
print(f"\nGenerating 12-month forecast...")
forecast = hw.forecast_future(periods=12)

forecast_df = pd.DataFrame(forecast)
print(forecast_df.to_string())

## 3. Model Comparison

In [ ]:
from services.model_comparison import ModelComparison

# Compare all models
mc = ModelComparison()
print("Comparing all 3 models...")
print("(This will take ~5 seconds)\n")

features = {
    'employee_counts': employee_counts,
    'start_date': '2023-01-01'
}

result = mc.compare_models('test_org', historical_data, forecast_horizon=12, features=features)

# Summary table
print("\n=== MODEL COMPARISON SUMMARY ===")
summary_table = mc.get_comparison_metrics()
for row in summary_table:
    print("\n" + "-" * 70)
    print(f"Model: {row['model']}")
    print(f"  RMSE: {row['rmse']:.4f}  |  MAE: {row['mae']:.4f}  |  MAPE: {row['mape']:.2f}%  |  R²: {row['r2']:.4f}")
    print(f"  Training Time: {row['training_time_seconds']:.3f}s")

print(f"\n\n*** BEST MODEL: {result['best_model'].upper()} ***")
print(f"(Selected based on lowest RMSE)")

## 4. Visualize Forecasts Side-by-Side

In [ ]:
# Get forecasts from each model
prophet_f = prophet.forecast_future(periods=12)
xgb_f = xgb.forecast_future(historical_data, periods=12, employee_counts=employee_counts)
hw_f = hw.forecast_future(periods=12)

# Extract predictions
prophet_pred = [f['predicted_emissions'] for f in prophet_f]
xgb_pred = [f['predicted_emissions'] for f in xgb_f]
hw_pred = [f['predicted_emissions'] for f in hw_f]

# Plot
plt.figure(figsize=(14, 6))

# Historical
plt.plot(range(36), historical_data, 'k-', linewidth=2.5, label='Historical', zorder=3)

# Forecasts
forecast_months = range(36, 48)
plt.plot(forecast_months, prophet_pred, 'b-', linewidth=2, label='Prophet', marker='o')
plt.plot(forecast_months, xgb_pred, 'g-', linewidth=2, label='XGBoost', marker='s')
plt.plot(forecast_months, hw_pred, 'r-', linewidth=2, label='Holt-Winters', marker='^')

# Styling
plt.axvline(x=35.5, color='gray', linestyle='--', alpha=0.5)
plt.text(17, max(historical_data) * 1.05, 'Historical', ha='center', fontsize=10, color='gray')
plt.text(41, max(historical_data) * 1.05, 'Forecast', ha='center', fontsize=10, color='gray')

plt.title('Model Forecast Comparison (12 months ahead)', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Emissions (kg CO2e)', fontsize=12)
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Statistics
print("\nForecast Statistics (12 months ahead):")
print(f"\nProphet: avg={np.mean(prophet_pred):.1f}, std={np.std(prophet_pred):.1f}")
print(f"XGBoost: avg={np.mean(xgb_pred):.1f}, std={np.std(xgb_pred):.1f}")
print(f"Holt-Winters: avg={np.mean(hw_pred):.1f}, std={np.std(hw_pred):.1f}")

## 5. Confidence Intervals

In [ ]:
# Extract confidence bounds
prophet_f = prophet.forecast_future(periods=12)

# Create DataFrame for better display
prophet_df = pd.DataFrame(prophet_f)
prophet_df['forecast_month'] = range(1, 13)

print("Prophet Forecast with 90% Confidence Intervals:\n")
print(prophet_df[['forecast_month', 'month', 'predicted_emissions', 'lower_bound', 'upper_bound']].to_string(index=False))

# Visualize with confidence bands
plt.figure(figsize=(12, 6))

forecast_months = range(1, 13)
pred = [f['predicted_emissions'] for f in prophet_f]
lower = [f['lower_bound'] for f in prophet_f]
upper = [f['upper_bound'] for f in prophet_f]

plt.plot(forecast_months, pred, 'b-', linewidth=2, marker='o', label='Predicted')
plt.fill_between(forecast_months, lower, upper, alpha=0.2, color='blue', label='90% Confidence Interval')

plt.title('Prophet Forecast with Confidence Bands', fontsize=14, fontweight='bold')
plt.xlabel('Forecast Month', fontsize=12)
plt.ylabel('Emissions (kg CO2e)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Testing with Real Data

When you have your own dataset, load it like this:

In [ ]:
# Example: Load from CSV
# df = pd.read_csv('your_emissions_data.csv')
# historical_data = df['total_emissions'].tolist()  # Must be 36+ months
# employee_counts = df['employee_count'].tolist()

# Then run:
# mc = ModelComparison()
# result = mc.compare_models('your_org_id', historical_data, forecast_horizon=12, features={'employee_counts': employee_counts})

print("\n=== REAL DATA TESTING ===")
print("""
Your CSV should have columns:
  - month (YYYY-MM format)
  - total_emissions (kg CO2e)
  - employee_count (optional, for normalization)
  
Minimum: 2 months of data
Recommended: 36+ months (3 years) for seasonal detection
""")

## 7. API Integration Example

In [ ]:
import requests
import json

# Make sure Flask server is running: python app.py

BASE_URL = 'http://localhost:5001/api/v1'

# 1. Check health
print("Checking ML Service health...")
try:
    response = requests.get('http://localhost:5001/health')
    print(f"Status: {response.status_code}")
    print(f"Response: {response.json()}")
except:
    print("ML Service not running. Start it with: python app.py")

# 2. Generate forecast (if service is running)
# print("\nGenerating forecast via API...")
# payload = {
#     'organization_id': 'test_org_123',
#     'historical_data': historical_data,
#     'months_ahead': 12,
#     'features': {
#         'employee_counts': employee_counts,
#         'start_date': '2023-01-01'
#     }
# }
# 
# response = requests.post(
#     f'{BASE_URL}/forecast/generate',
#     json=payload
# )
# 
# print(f"Status: {response.status_code}")
# result = response.json()
# print(f"Best model: {result['best_model']}")